In [2]:
import pandas as pd
import numpy as np
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool
import warnings
warnings.filterwarnings('ignore')

# 1. LOAD DATA
train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

# 2. HYBRID FEATURE ENGINEERING
def engineer_features(train_df, test_df):
    train_df['is_test'] = 0
    test_df['is_test'] = 1
    df = pd.concat([train_df, test_df], ignore_index=True)

    combine_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps',
                    'Broad_Jump', 'Agility_3cone', 'Shuttle']

    # Missingness Indicators
    for col in combine_cols:
        df[f'{col}_ismissing'] = df[col].isnull().astype(int)

    # Biomechanical Composites
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)
    df['Explosiveness'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Agility_Score'] = df['Agility_3cone'] + df['Shuttle']
    df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)

    # Positional Relativism
    for col in combine_cols + ['Height', 'Weight', 'BMI']:
        df[f'{col}_Pos_Mean'] = df.groupby('Position')[col].transform('mean')
        df[f'{col}_Pos_Diff'] = df[col] - df[f'{col}_Pos_Mean']

    # Categorical Preparation (NO Label Encoding - leaving for CatBoost Native Handling)
    cat_cols = ['School', 'Player_Type', 'Position_Type', 'Position']
    for col in cat_cols:
        df[f'{col}_Freq'] = df.groupby(col)['Id'].transform('count')
        # Cast to string and fill missing with 'Unknown' so CatBoost can process them
        df[col] = df[col].fillna('Unknown').astype(str)

    train_out = df[df['is_test'] == 0].drop(['is_test'], axis=1)
    test_out = df[df['is_test'] == 1].drop(['is_test', 'Drafted'], axis=1)
    return train_out, test_out

print("Engineering hybrid features...")
train_df, test_df = engineer_features(train, test)

features = [c for c in train_df.columns if c not in ['Id', 'Drafted']]
cat_features = ['School', 'Player_Type', 'Position_Type', 'Position']

X = train_df[features]
y = train_df['Drafted']
X_test = test_df[features]

# 3. ACTIVE OPTUNA HYPERPARAMETER TUNING (CATBOOST)
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 5.0),
        'eval_metric': 'AUC',
        'verbose': 0,
        'random_seed': 42
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_scores = []

    for tr_idx, va_idx in cv.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        # Wrapping data in CatBoost Pools is required to pass cat_features
        train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
        val_pool = Pool(X_va, y_va, cat_features=cat_features)

        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)

        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))

    return np.mean(auc_scores)

print("Starting Optuna optimization for CatBoost...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20) # Increase n_trials if you have time before submission
best_params = study.best_params
print(f"\nBest Optuna Parameters Found: {best_params}")

# 4. FINAL CROSS-VALIDATION & INFERENCE WITH BEST PARAMS
best_params['eval_metric'] = 'AUC'
best_params['verbose'] = 0
best_params['random_seed'] = 42

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

print("\nTraining Final CatBoost Model with Native Categoricals...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_va, y_va, cat_features=cat_features)
    test_pool = Pool(X_test, cat_features=cat_features)

    model = CatBoostClassifier(**best_params)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)

    oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]
    fold_auc = roc_auc_score(y_va, oof_preds[val_idx])
    print(f"Fold {fold+1} ROC-AUC: {fold_auc:.5f}")

    test_preds += model.predict_proba(test_pool)[:, 1] / skf.n_splits

print(f"\n[!] FINAL OUT-OF-FOLD ROC-AUC: {roc_auc_score(y, oof_preds):.5f}")

# 5. SUBMISSION
submission = test[['Id']].copy()
submission['Drafted'] = test_preds
submission.to_csv('Data/submission_2.csv', index=False)
print("Hybrid master predictions saved to 'Data/submission_hybrid_master.csv'")

[I 2026-05-21 20:35:47,792] A new study created in memory with name: no-name-c1a4d29f-c4ea-42ca-8764-a8a3fd796dc4


Engineering features...
[LightGBM] [Info] Number of positive: 1442, number of negative: 782
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000530 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4245
[LightGBM] [Info] Number of data points in the train set: 2224, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.648381 -> initscore=0.611932
[LightGBM] [Info] Start training from score 0.611932
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

[I 2026-05-21 20:36:25,062] Trial 0 finished with value: 0.8110159314809099 and parameters: {'n_estimators': 321, 'learning_rate': 0.060663962811555006, 'max_depth': 5, 'subsample': 0.7895924027142688, 'colsample_bytree': 0.8949070569587606}. Best is trial 0 with value: 0.8110159314809099.


[LightGBM] [Info] Number of positive: 1442, number of negative: 782
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001048 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4245
[LightGBM] [Info] Number of data points in the train set: 2224, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.648381 -> initscore=0.611932
[LightGBM] [Info] Start training from score 0.611932
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

[I 2026-05-21 20:36:58,008] Trial 1 finished with value: 0.8139291675545138 and parameters: {'n_estimators': 810, 'learning_rate': 0.061949134044324575, 'max_depth': 8, 'subsample': 0.9745321168717691, 'colsample_bytree': 0.7351086784569099}. Best is trial 1 with value: 0.8139291675545138.


...

[LightGBM] [Info] Number of positive: 1442, number of negative: 782
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000638 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4245
[LightGBM] [Info] Number of data points in the train set: 2224, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.648381 -> initscore=0.611932
[LightGBM] [Info] Start training from score 0.611932
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

[I 2026-05-21 20:37:30,941] Trial 18 finished with value: 0.8223406181493835 and parameters: {'n_estimators': 447, 'learning_rate': 0.02216355314954734, 'max_depth': 6, 'subsample': 0.683914946150908, 'colsample_bytree': 0.6550922660073273}. Best is trial 10 with value: 0.8315660929120565.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

[I 2026-05-21 20:37:31,781] Trial 19 finished with value: 0.8304708507097704 and parameters: {'n_estimators': 433, 'learning_rate': 0.01369693642673671, 'max_depth': 4, 'subsample': 0.8642658790894854, 'colsample_bytree': 0.7098620237596127}. Best is trial 10 with value: 0.8315660929120565.


Best parameters: {'n_estimators': 525, 'learning_rate': 0.010066413396113858, 'max_depth': 4, 'subsample': 0.8270532551768642, 'colsample_bytree': 0.6071653409797987}

Commencing 5-Fold Stratified Cross-Validation Ensemble...
[LightGBM] [Info] Number of positive: 1442, number of negative: 782
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000323 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4245
[LightGBM] [Info] Number of data points in the train set: 2224, number of used features: 46
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.648381 -> initscore=0.611932
[LightGBM] [Info] Start training from score 0.611932
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be